# Phase 2: exploratory bigrams and trigrams

Find recurring sequences in the code-switched subset to support contextual analysis of sentiment. A phrase from a code-switched review is not necessarily bilingual. No automatic phrase-language or phrase-sentiment classification is performed.

Rules: lowercase token copies; no stop-word removal, stemming or spelling correction. Retain negation, contrast markers and informal forms such as betul2. Internal apostrophes and hyphens stay in tokens; curly apostrophes are normalised. Decimal numbers remain tokens. Boundaries are full stops (except between digits), question/exclamation marks, common Unicode equivalents, ellipses, semicolons and line breaks. Abbreviation full stops may over-segment. Commas, colons, brackets and quote/dash separators do not break adjacency; other symbols, including emojis, break token runs. Thus extraction stays within punctuation-defined segments and token runs, never crossing reviews. This conservative heuristic is not a linguistic sentence parser.

Count overlapping two/three-token windows. Rank by occurrences, distinct reviews, then alphabetically. Recurring means at least two distinct reviews. Food names remain in frequency counts but alone do not prove switching. Sentiment breakdowns refer to whole-review labels, not phrase polarity. Run from the repository root or a child directory with pandas/openpyxl. Source annotations remain unchanged.


In [1]:
from pathlib import Path
import re, hashlib
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Data/CPC 353 Dataset.xlsx').exists())
SOURCE = ROOT/'Data/CPC 353 Dataset.xlsx'
OUT = ROOT/'Outputs/Tables'
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
df = pd.read_excel(SOURCE, sheet_name='Self-Collected Data', header=3, usecols='A:M', keep_default_na=False)
populated = df.apply(lambda c: c.astype(str).str.strip().ne('')).any(axis=1)
df.insert(0,'source_row',range(5,len(df)+5))
df=df.loc[populated].copy()
assert df['Review ID'].is_unique
assert df['Review Text'].astype(str).str.strip().ne('').all()
cs=df.loc[df['Language Classification'].eq('Code-switched')].copy()
assert not cs.empty
assert cs['Sentiment'].isin(['Positive','Negative','Neutral']).all()
print('All reviews:',len(df),'Code-switched reviews:',len(cs))


All reviews: 197 Code-switched reviews: 144


In [2]:
BOUNDARY = re.compile(r'(?<!\d)\.|\.(?!\d)|[!?…‼⁉。！？;；\r\n]+')
TOKEN = re.compile(r"\d+(?:[.,]\d+)+|[^\W_]+(?:['’\-][^\W_]+)*", re.UNICODE)
GAP = re.compile(r"[\s,:，：()\[\]{}\"'‘’“”\-–—]*")
def token_runs(text):
    start=0
    for sid,boundary in enumerate([*BOUNDARY.finditer(text),None],1):
        stop=boundary.start() if boundary else len(text)
        run=[]; rid=1
        for token in TOKEN.finditer(text,start,stop):
            if run and not GAP.fullmatch(text[run[-1].end():token.start()]):
                yield sid,rid,start,stop,run
                run=[];rid+=1
            run.append(token)
        if run:yield sid,rid,start,stop,run
        start=boundary.end() if boundary else len(text)
def extract_ngrams(text,sizes=(2,3)):
    for sid,rid,start,stop,tokens in token_runs(text):
        for n in sizes:
            for i in range(len(tokens)-n+1):
                w=tokens[i:i+n]
                yield dict(n=n,ngram=' '.join(t.group().lower().replace('’',"'") for t in w),segment_id=sid,run_id=rid,match_start=w[0].start(),match_end=w[-1].end(),source_quote=text[w[0].start():w[-1].end()],segment_text=text[start:stop])
rows=[]
for _,r in cs.iterrows():
    for h in extract_ngrams(r['Review Text']):
        rows.append(dict(review_id=r['Review ID'],source_row=r['source_row'],sentiment=r['Sentiment'],primary_switching_type=r['Code-Switching Type'],**h))
hits=pd.DataFrame(rows)
assert not hits.empty
assert not hits.duplicated(['review_id','n','match_start','match_end']).any()
lookup=cs.set_index('Review ID')['Review Text']
for h in hits.itertuples():assert lookup.loc[h.review_id][h.match_start:h.match_end]==h.source_quote
print(hits.groupby('n').size().to_string())


n
2    7610
3    6707


In [3]:
summary=hits.groupby(['n','ngram']).agg(occurrences=('review_id','size'),distinct_reviews=('review_id','nunique')).reset_index().sort_values(['n','occurrences','distinct_reviews','ngram'],ascending=[True,False,False,True])
summary['review_percentage']=(100*summary['distinct_reviews']/len(cs)).round(2)
summary['review_denominator']=len(cs)
recurring=summary.loc[summary['distinct_reviews'].ge(2)].copy()
top=summary.groupby('n',group_keys=False).head(20)
by_sentiment=hits.groupby(['n','ngram','sentiment']).agg(occurrences=('review_id','size'),distinct_reviews=('review_id','nunique')).reset_index()
by_sentiment['review_denominator']=by_sentiment['sentiment'].map(cs['Sentiment'].value_counts())
by_sentiment['within_sentiment_review_percentage']=(100*by_sentiment['distinct_reviews']/by_sentiment['review_denominator']).round(2)
assert summary['occurrences'].sum()==len(hits)==by_sentiment['occurrences'].sum()
assert summary['distinct_reviews'].le(summary['occurrences']).all()
print(top[['n','ngram','occurrences','distinct_reviews']].to_string(index=False))


 n              ngram  occurrences  distinct_reviews
 2         ayam gepuk           60                48
 2             it was           14                 8
 2           kedai ni           13                10
 2         first time           12                 9
 2          puas hati           11                10
 2            this is           11                 9
 2             of the           11                 8
 2             on the           11                 6
 2       kobis goreng           10                 9
 2       kubis goreng           10                 8
 2         the sambal           10                 7
 2       boleh tambah            9                 9
 2           ayam dia            9                 7
 2            dine in            8                 8
 2         and sambal            8                 7
 2            and the            8                 7
 2      gepuk artisan            8                 7
 2        tambah nasi            8            

In [4]:
def phrases(text):return [h['ngram'] for h in extract_ngrams(text,(2,))]
assert phrases('sedap. service bagus')==['service bagus']
assert phrases('sedap\nservice bagus')==['service bagus']
assert phrases('sedap‼ service bagus')==['service bagus']
assert phrases('tak sedap but not bad')==['tak sedap','sedap but','but not','not bad']
assert phrases('very very very')==['very very','very very']
assert phrases('sedap 😊 service bagus')==['service bagus']
assert phrases('ayam, juicy')==['ayam juicy']
assert phrases("don't regret")==["don't regret"]
assert phrases('harga 9.90 murah')==['harga 9.90','9.90 murah']
assert phrases('check betul2')==['check betul2']
assert phrases('one')==[]
for table,name in [(hits,'ngram_occurrences.csv'),(summary,'ngram_frequencies.csv'),(recurring,'ngram_recurring.csv'),(top,'ngram_top20.csv'),(by_sentiment,'ngram_by_review_sentiment.csv')]:
    table.to_csv(OUT/name,index=False,encoding='utf-8-sig')
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==source_hash
print('Boundary, negation, overlap, token and source-offset checks passed; source unchanged.')
print('Recurring phrases:',recurring.groupby('n').size().to_dict())


Boundary, negation, overlap, token and source-offset checks passed; source unchanged.
Recurring phrases: {2: 608, 3: 99}
